> Generate mask from image and create processed image if it is availble

In [ ]:
#| default_exp mask_generation

In [ ]:
#| export
from pathlib import Path
import yaml
from fastcore.all import *
import numpy as np
from tqdm.auto import tqdm
from argparse import ArgumentParser
import cv2

In [ ]:
#| export
import tensorflow as tf

In [ ]:
#| export

path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs')
custom_lib_path=Path(r'/home/ai_warstein/custom_libs/goni/custom_libs')
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
dotenv_p = load_dotenv(Path(Path(path).parent, 'private_easy_pin_detection/.env'))
CV_TOOLS = os.getenv('CV_TOOLS')
PRIVATE_EASY_PIN_DETECTION = os.getenv('PRIVATE_EASY_PIN_DETECTION')
sys.path.append(CV_TOOLS)
sys.path.append(PRIVATE_EASY_PIN_DETECTION)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| exporti
from private_easy_pin_detection.three_channel_dataset import *

In [ ]:
config_path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection')

In [ ]:
with open(f'{config_path}/config_office_test.yaml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
#| export
def predict_msk(
        model:tf.keras.Model,
        config:dict,
        im_file:str,
        save_path:str=None
  ):

  img = read_normalize(im_file=im_file, config=config)
  pred = model.predict(img[None,...])
  msk = pred[0] > config['model']['threshold']
  msk = msk * 255.0
  msk = msk.astype(np.uint8)
  name_= Path(im_file).name
  if save_path:
    cv2.imwrite(f'{save_path}/{name_}', msk)
  return msk

In [ ]:
#| export
def load_model(config):
    model_config = config['model']
    model_f_n = f"{model_config['save_path']}/{model_config['model_name']}"
    model = tf.keras.models.load_model(
        model_f_n,
        compile=False)
    return model

In [ ]:
#| export
def parse_args_():
    'Parse all the arguments for mask creation '
    parser = ArgumentParser()
    parser.add_argument(
        '-is',
        '--image_source', 
        type=str, 
        required=True, 
        help='image source directory')
    parser.add_argument(
        '-m',
        '--mask_path', 
        type=str, 
        required=True, 
        help='where the mask needs to be saved'
        )
    parser.add_argument(
        '-mn',
        '--model_name', 
        type=str, 
        required=True, 
        help='name of the save')
    parser.add_argument(
        '-mp',
        '--model_path', 
        type=str, 
        required=True, 
        help='model path')
    parser.add_argument(
        '-cp', 
        '--config_path',
        help='Config path file name with full path',
        )
    return parser.parse_args()


In [ ]:
#| export
def main_():
    'Together all the function so that we can use python script'
    args = parse_args_()

    with open(f'{args.config_path}', 'r') as f:
        config = yaml.safe_load(f)
    
    im_path = Path(args.image_source)
    model = load_model(config)

    fn = len(im_path.ls())
    for i in tqdm(im_path.ls(), total=fn):
        predict_msk(
            model=model,
            config=config,
            im_file=f'{i}',
            save_path=args.mask_path
        )


    
    

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('18_gray_scale_mask_generation.ipynb')